In [ ]:
from scipy import interpolate
from scipy.optimize import curve_fit
from tqdm import tqdm
import itertools as it
import seaborn as sns
import json
from scipy.signal import savgol_filter
import math
from modelbase.ode import Simulator
from collections.abc import Iterable
from typing import Union

# generalized function for both DIRKs and PIRKs

In [ ]:
def apply_protocol(s: Simulator, protocol, protocol_time_is_accumulated:bool = False,
        use_tqdm: bool = True, shift_time:bool = False, input_in_ms = False, output_in_ms:bool = False):
    """
    DIRK:
    illumination -> dark (relaxation) -> illumination
    simulator has to be initialised
    PIRK:
    steady state (light) -> dark -> light pulses (fast) in dark period
    this time without strange illumination before the experiment.

    the first element has to be some form of pre-illumination. only then, the "main thing" should start.

    Protocol has to be provided in seconds!
    """
    protocol = list(protocol)

    if not protocol_time_is_accumulated:
        time, light = zip(*protocol)
        time = it.accumulate(time)
        protocol = list(zip(time, light))

    if input_in_ms: #, then we have to convert them to seconds for modelbase
        timevalues, pfds = zip(*protocol)
        protocol = list(zip(np.array(timevalues)/1000, pfds))

    # print(protocol)

    pre_protocol_time = protocol[0][0]
    pre_protocol_pfd = protocol[0][1] # the first pfd value defined in protocol

    y0 = get_stst_y0(s, pfd=pre_protocol_pfd)
    if y0 is None:
        raise ValueError("Modelbase says NO")
    s.initialise(y0)

    if use_tqdm: protocol = tqdm(protocol)

    for t, pfd in protocol:
        # print(f"Time: {t}, PFD: {pfd}")
        s.update_parameter("pfd", pfd)
        s.simulate(t)

    c = s.get_full_results_df()
    v = s.get_fluxes_df()

    if shift_time:
        c.index = c.index - pre_protocol_time # shift the index such that event starts at 0
        v.index = v.index - pre_protocol_time # shift the index such that event starts at 0
    if output_in_ms: # then we convert (back) from seconds to milliseconds
        c.index = c.index * 1000
        v.index = v.index * 1000

    return c, v

# Trace analysis

## curve_fitting

In [ ]:
def exponential_decay(x, A, B, Tau):
    return A * np.exp(-x / Tau) + B   # B = baseline, A = amplitude, Tau = decay constant

def exponential_decay_fix_origin(x, A, Tau):
    B = -A # enforce y=0 for x=0
    return A * np.exp(-x / Tau) + B   # B = baseline, A = amplitude, Tau = decay constant

def linear(x, m, b):
    return m * x + b  # m = slope, b = intercept

def fit_function(x, y, model_func, p0, bounds=None, visualize: Union[str, float] = "", title=""):
    try:
        if bounds is None:
            inf = np.inf
            bounds = ([-inf] * len(p0), [inf] * len(p0))

        p_opt, p_cov = curve_fit(model_func, x, y,
                                p0=p0,
                                bounds=bounds,
                                maxfev=10000)

        fitted_curve = model_func(x, *p_opt)

        # R² calculation
        residuals = y - fitted_curve
        ss_res = np.sum(residuals**2)
        ss_tot = np.sum((y - np.mean(y))**2)
        R2 = 1 - (ss_res / ss_tot)

        if visualize == "all" or isinstance(visualize, float):
            cutoff = 1 if visualize == "all" else visualize
            if R2 <= cutoff:
                fig, ax = plt.subplots(figsize=(5, 3))
                ax.set(title=title, xlabel="time (ms)")
                ax.plot(x, y, label='Data')
                ax.plot(x, fitted_curve, color='red', label=f'Fit (R² = {R2:.3f})')
                ax.legend()
                fig.show()

        out = list(p_opt) + [R2]

    except Exception as e:
        out = [np.nan] * (len(p0) + 1)
        raise RuntimeError(f"Error in fitting: {e}")

    return out

### PhotosnQ macro function analogues

In [ ]:
def GetProtocolByLabel_from_JSON(label, json_file):
    meta_information = json_file[0]
    protocol_and_script_outputs = meta_information['sample'][0]
    set_of_measurements = protocol_and_script_outputs["set"]
    out = [protocol for protocol in set_of_measurements if protocol.get('label', "") == label]
    return out

def transform_trace(op, trace, value):
    trace = np.array(trace)
    value = np.array(value)  # scalar or array
    
    if op == 'add':         return (trace + value).tolist()
    elif op == 'divide':    return (trace / value).tolist()
    elif op == 'abs':       return (-np.log10(trace / value)).tolist()
    elif op == 'subtract':  return (trace - value).tolist()
    else:
        raise ValueError(f"Unknown operation: {op}")

In [ ]:
def get_autogain_by_idx(autogain_list, index):
    return [element for element in autogain_list if element[0] == index]

# DIRK - trace analysis

In [ ]:
def analyse_DIRK_from_simulation(c_given, protocol:Iterable, input_is_in_ms = False, protocol_time_is_accumulated=False, visualize: Union[str, float]="", correct_for_amplitude = False):
    """
    wrapper function to analyse the DIRK traces from the simulation. Since it is from the simulation and has access to c (the concentrations), it can analyse ECS and P700 simulatenously.
    NOTE: make sure protocol and c.index both have the same unit of time! (either ms or s!)
    """
    c = c_given.copy()

    # Fix time units
    if not input_is_in_ms: # then make it in ms
        c.index = c.index * 1000
        timevalues, pfds = zip(*protocol)
        protocol = list(zip(np.array(timevalues)*1000, pfds))

    # Protocol time construction
    if not protocol_time_is_accumulated: # then accumulate it
        durations, pfds = zip(*protocol)
        onset_times = list(it.accumulate(durations))
        protocol = list(zip(onset_times, pfds))
    else:
        onset_times, pfds = zip(*protocol)

    PAR = protocol[0][1]

    t_start, t_end = onset_times[0], onset_times[1]
    # print(t_start, t_end)
    # Convert t_point values of dar_period to closest index positions (gets the correct index reliably)
    idx_start = (np.abs(c.index - t_start)).argmin()
    idx_end = (np.abs(c.index - t_end)).argmin()

    data = c.iloc[idx_start:idx_end+1]
    
    x = data.index.to_numpy()
    # because the data might not be evenly spaced, some regions might impact the fit stronger than others. To ensure a fit were the complete behaviour contributes equally, we will interpolate the data to evenly spaced x values.
    x_even = np.linspace(min(x), max(x), 1000, endpoint=False)
    x_even_for_fit = x_even - min(x_even) # make it start at 0!
    # print(x_even, min(x_even))

    ##############################################################
    # analyse the ECS dynamics! ###################################

    y = 1e6*data["dpsi"].to_numpy() # in the model, psi is in kV (see matuszynska scirpt for reason), but here we convert to mV to have nice numbers.

    y_even = interpolate.interp1d(x, y)(x_even) # interpolate y values using a linear interpolation to match x_even, interp1d returns a function that receives x_even.
    # we interpolate because from the simulation we get an uneven coverage of x-values, such that dense regions would influence the fit much more than sparse ones. with interpolation we equilibrate everything 

    (ECS_tot, B_ECS, Tau_ECS, ECS_Rsquared)  = fit_function(x_even_for_fit, y_even, exponential_decay, #a = amplitude, b = baseline, gH = decay constant = proton conductance
                                    p0 = [10, 0, 5.6], bounds=([0, 0, 0], [100, 50, 25]),
                                    visualize= visualize, title=r'$\Delta \psi$ - ' + f"{PAR}" + r' $\mu$E')
    if correct_for_amplitude: ECS_tot = y[0] - B_ECS

    fitting_input_vs_time_ECS = pd.Series(y_even, index = x_even_for_fit)
    
    gH = 1 / Tau_ECS # proton conductance
    vH = ECS_tot / Tau_ECS # proton flux

    ###############################################################
    # analyse the P700 dynamics! ###################################

    y = data["rel_P700+"].to_numpy() # relative P700+ signal
    y_even = interpolate.interp1d(x, y)(x_even) # interpolate y values using a linear interpolation to match x_even
    (P700plus_tot, B_P700, Tau_P700, P700_Rsquared)  = fit_function(x_even_for_fit, y_even, exponential_decay,
                                    p0 = [0.002, 0, 20], bounds=([0, -np.inf, 0], [np.inf, np.inf, np.inf]),
                                    visualize= visualize, title='P700$^+$ - ' + f"{PAR}" + r' $\mu$E') #a = amplitude, b = baseline, gH = decay constant = P700 conductance
    if correct_for_amplitude: P700plus_tot = y[0] - B_P700

    fitting_input_vs_time_P700 = pd.Series(y_even, index = x_even_for_fit)

    k_PSI = 1 / Tau_P700
    vPSI = P700plus_tot / Tau_P700

    result = {
        "ECS_tot": ECS_tot,
        "ECS_b": B_ECS, 
        "gH": gH,
        "vH": vH,
        "ECS_Rsquared": ECS_Rsquared,
        "P700plus_tot": P700plus_tot,
        "P700_b": B_P700,
        "k_PSI": k_PSI,
        "vPSI": vPSI,
        "P700_Rsquared": P700_Rsquared,
        "traces": {"P700": fitting_input_vs_time_P700, "ECS": fitting_input_vs_time_ECS}
    }

    return result


In [ ]:
def analyze_DIRK_from_MultispeQ(json_data, Protocol_label, remove_spike_artifacts = False, use_fraction_of_relaxation_window = 1, show_fits = "", bool_make_absorbances_positive = False):
    """
    This function simulates the readout from the in_vivo measurements. It can analyse both ECS and P700 (because we essentially extract the same features), but not simultaneously so. This is because as of now, I measure all of them separately.
    ASSUMPTIONS about the protocol:
     "v_arrays": [
       [
         1000,  <-- the time distance between the measurement pulses in µs (constant - I would have used save_trace_time_scale but it does not work)
         150,   <-- the number of pulses before and after the dark period
         30     <-- the number of pulses in the dark period. determines the length of the dark-period.
         2      <-- index of autogain used (important for python-analysis!) 
       ],
       [
         50,    <-- all the PARs tested (sequentially after another)
         100,
         500,
         ...
       ]
     ],
    This function is designed specifically to work with json files generated by "AP_DIRK_ECS_only" and "AP_DIRK_P700(_only)"
    """
    # Extract data from JSON
    meta_information = json_data[0]
    protocol_and_script_outputs = meta_information['sample'][0]
    set_of_measurements = protocol_and_script_outputs["set"]
    data = GetProtocolByLabel_from_JSON(Protocol_label, json_data)

    # Constants and assumptions
    trace_begin = protocol_and_script_outputs['v_arrays'][0][1]
    length_baseline = protocol_and_script_outputs['v_arrays'][0][1]
    pulses_dark = protocol_and_script_outputs['v_arrays'][0][2]
    recovery = protocol_and_script_outputs['v_arrays'][0][1]

    # we have to define dt to create a time axis
    pulse_distance = protocol_and_script_outputs['v_arrays'][0][0] / 1000  # µs → ms

    index_autogain_used = protocol_and_script_outputs['v_arrays'][0][3]
    autogaines_defined = set_of_measurements[0]['autogain']
    autogain_used = get_autogain_by_idx(autogaines_defined, index_autogain_used)[0]
    pulse_duration = autogain_used[3] / 1000 # 3: the index in which the duration is stored # 1000: µs -> ms

    dt = pulse_distance + pulse_duration

    num_subtraces = 6
    end_dark = length_baseline + pulses_dark
    len_subtrace = length_baseline + pulses_dark + recovery
    total_pulses = trace_begin + num_subtraces * len_subtrace
    pulses_for_fit = math.floor(pulses_dark * use_fraction_of_relaxation_window)
    PAR_levels = protocol_and_script_outputs['v_arrays'][1]

    # Construct time axis for fit
    t_ax_exp_decay_fit = np.array([i * dt for i in range(pulses_for_fit + 1)])
    t_ax_complete_subtrace = np.array([i * dt for i in range(len_subtrace)])

    # Placeholder for output
    results = []

    # Main loop over PAR levels
    for par_index, PAR in enumerate(PAR_levels):
        trace_raw = data[par_index]['data_raw']
        trace = trace_raw[trace_begin: total_pulses]

        # Average subtraces
        avg_trace = trace[0:len_subtrace]
        for i in range(1, num_subtraces):
            next_trace = trace[i * len_subtrace: (i + 1) * len_subtrace]
            avg_trace = transform_trace('add', avg_trace, next_trace)
        avg_trace = transform_trace('divide', avg_trace, num_subtraces)

        if remove_spike_artifacts:  # needed for delta_psi, but not for P700
            avg_trace[length_baseline] = np.mean([avg_trace[length_baseline - 1], avg_trace[length_baseline + 1]])
            avg_trace[end_dark] = np.mean([avg_trace[end_dark - 1], avg_trace[end_dark + 1]])
            avg_trace[0] = np.mean([avg_trace[1], avg_trace[2]])

        # Absorbance transformation
        if bool_make_absorbances_positive:
            baseline_val = np.max(avg_trace[int(0.05 * length_baseline):int(0.95 * length_baseline)]) *1.1 # this RUINS fitting unless initials and bounds are adapted!
        else:
            baseline_val = np.mean(avg_trace[int(0.05 * length_baseline):int(0.95 * length_baseline)])
        absorbance = transform_trace('abs', avg_trace, baseline_val)

        # convert from Absorbtion units to milli-absorbtion units
        absorbance = np.array(absorbance)*1000

        # Prepare window for fit
        y_fit = absorbance[length_baseline - 1:length_baseline + pulses_for_fit]

        # Fit the decay model, here with much better options!
        A, B, Tau, R2 = fit_function(t_ax_exp_decay_fit, y_fit, exponential_decay,
                                    p0 = [1.5, -1.5, 5.6], bounds=([0, -np.inf, 0], [np.inf, 10, 25]), # notice these values (those that are not 0 or inf). they are somewhat arbitrary and have been chosen based on my experience. 
                                    visualize= show_fits, title=f'Fit for PAR = {PAR}')

        # create a pandas series that enables much easier plotting later
        absorbance_vs_time = pd.Series(absorbance, index = t_ax_complete_subtrace)
        fitting_input_vs_time = pd.Series(y_fit, index = t_ax_exp_decay_fit)

        meta_info_for_sim_of_this_protocol = [[dt*(length_baseline), PAR],
                                              [dt*(length_baseline + pulses_dark), 0],
                                              [dt*(2 * length_baseline + pulses_dark), PAR]]

        results.append({
            'PAR': PAR,
            'amplitude': A,
            'baseline': B,
            'Tau': Tau,
            'rate': 1 / Tau,
            'flux':(1 / Tau) * A,
            'r_squared': R2,
            'traces': {"absorbance": absorbance_vs_time, "fitting_input": fitting_input_vs_time},
            "simulation_meta_info": meta_info_for_sim_of_this_protocol
        })
    
    results = pd.DataFrame(results)

    # print some helpful information
    print(f"length of dark period: {t_ax_exp_decay_fit[-1]} ms\ntemporal resolution: {dt} ms")

    return t_ax_exp_decay_fit, results

# PIRK - trace analysis

In [ ]:
import numpy as np

def analyze_PIRK_from_MultispeQ(json_data, Protocol_label, datapoints_to_use = np.nan, show_abs_traces = False, show_fits: Union[str, float]= "", use_fixed_fit_wndw_len = True):
    """
    This function simulates the readout from the in_vivo measurements. It can analyse both ECS and P700, but not simultaneously so. This is because as of now, I measure all of them separately.

     ASSUMPTIONS about the protocol:
     "v_arrays": [
       [
         1000, <-- pulse distance
         2500 <-- pulse brighness (ppfd) in µEm-2s-1
       ],
       [
         40, <-- The very first is the baseline and continuation of the pre-illumination.
         2,  <-- then the other number of pulses of every phase. 2 are the pulsed time frames. 
         40, <-- here we have 40 pulses dark
         2,  <-- and so on...
         60,
         ...
       ],
       [
         100,  <-- all PARs to be tested.
         500,
         1000,
         5000
       ]

    peak_hight and amplitude (if both are still in this function) are the same basically. peak hight is just measured as a simple difference, whereas amplitude comes from the exponential fit. I plan on getting rid of one of them, whatever seems to be more robust.
    """

    # Extract data from JSON
    # first we subset different sections
    meta_information = json_data[0]
    protocol_and_script_outputs = meta_information['sample'][0]
    set_of_measurements = protocol_and_script_outputs["set"]

    pulse_distance = protocol_and_script_outputs['v_arrays'][0][0] / 1000  # µs → ms
    pulse_brightness = protocol_and_script_outputs['v_arrays'][0][1]

    # we have to define dt to create a time axis
    index_autogain_used = protocol_and_script_outputs['v_arrays'][0][2]
    autogaines_defined = set_of_measurements[0]['autogain']

    autogain_used = get_autogain_by_idx(autogaines_defined, index_autogain_used)[0]
    pulse_duration = autogain_used[3] / 1000 # 3: index of autogain duration definition, #1000: µs -> ms

    dt = pulse_distance + pulse_duration # in ms

    # now the real structural information of the protocol
    pulse_protocol = protocol_and_script_outputs['v_arrays'][1]
    protocol = list((it.accumulate(pulse_protocol)))
    
    PAR_levels = protocol_and_script_outputs['v_arrays'][2]
    
    data = GetProtocolByLabel_from_JSON(Protocol_label, json_data)
    
    t_ax_complete = np.array([i * dt for i in range(0,protocol[-1]+1)])
    #print(t_ax_complete)
    protocol_in_units_of_time = (np.array(protocol)*dt) # in ms

    print(f"protocol indeces: {protocol}")

    if use_fixed_fit_wndw_len:
      if datapoints_to_use is np.nan:
        number_values_to_consider_for_fit = min(x for x in pulse_protocol[1:] if x != min(pulse_protocol)) # the real min(protocol) is the peak_window_length, the second lowest (fetched here) is the shortest relaxation window. also we have to exclude the first value which is still at stst pfd and not relaxation in darkness
        print(f"shortest window length detected: {number_values_to_consider_for_fit}")
      else:
        number_values_to_consider_for_fit = datapoints_to_use
        # by using the same number of points per fit we avoid that the results change because of the fitting window size (an effect I have actually observed)
    else:
       number_values_to_consider_for_fit = np.inf


    # Placeholder for output
    results = []

    if show_abs_traces: fig_all, ax_all = plt.subplots(1,1, figsize=(10,5))

    # Main loop over PAR levels
    for par_index, PAR in enumerate(PAR_levels):
        trace = data[par_index]['data_raw']
        
        # calculate absorbance trace
        trace[0] = np.nan # get rid of some weird artifact
        trace = np.append(np.nan, trace) # we have no measurement at t=0, therefore we insert NAN

        baseline_val = np.mean(trace[2:int(0.95 * protocol[0])])  # use the first 95% of the first window as baseline        
        absorbance = transform_trace('abs', trace, baseline_val)

        # convert from Absorbtion units to milli-absorbtion units
        absorbance = np.array(absorbance)*1000

        absorbance_vs_time = pd.Series(absorbance, index = t_ax_complete)
        if show_abs_traces: absorbance_vs_time.plot(xlabel = "time (ms)", ax = ax_all, label= f"{PAR}")

        t_points, peak_hights, A, B, Tau, R2 = [], [], [], [], [], []

        pfd_values_for_simulation_of_protocol = []

        # loop through the post-puls relaxation
        for i, index in enumerate(protocol):
            if i % 2 == 0: # this is the end of a relaxation window / the beginning of a pulse
                pfd_values_for_simulation_of_protocol += [PAR] if i == 0 else [0]
                continue   
            
            pfd_values_for_simulation_of_protocol += [pulse_brightness]

            start_idx = index
            end_idx = protocol[i + 1]+1 if i + 1 < len(protocol) else len(trace)+1
            last_point_for_fitting = min(end_idx, start_idx + number_values_to_consider_for_fit+1)

            #low-tech amplitude calculations:
            end_idx_prev_window = protocol[i - 1]
            peak_hight = absorbance[start_idx] - np.mean(absorbance[end_idx_prev_window-2:end_idx_prev_window])

            #create and prepare x and y
            current_t = t_ax_complete[start_idx]

            x = t_ax_complete[start_idx:last_point_for_fitting] - current_t  # t starts at 0
            y = absorbance[start_idx:last_point_for_fitting]
            # print(f"last value in time axis: {x[-1]}\tcurrent_t = {current_t}\nstart_idx = {start_idx}\t end_idx = {end_idx}\tlast_p_fit = {last_point_for_fitting}")

            A_, B_, Tau_, R2_ = fit_function(x, y, exponential_decay,
                                         p0 = [1.5, -1.5, 5.6], bounds=([0, -np.inf, 0], [np.inf, 10, 25]), # notice these values (those that are not 0 or inf). they are somewhat arbitrary and have been chosen based on my experience. 
                                         visualize = show_fits, title = f'PAR = {PAR}, relaxation kinetic = {int((i+1)/2)}')
            
            t_points.append(current_t)
            A.append(A_)
            B.append(B_)
            Tau.append(Tau_)
            R2.append(R2_)
            peak_hights.append(peak_hight)

        A, Tau, = np.array(A), np.array(Tau)

        meta_info_for_sim_of_this_protocol = list(zip(protocol_in_units_of_time, pfd_values_for_simulation_of_protocol))
        # print(meta_info_for_sim_of_this_protocol)

        results.append({
            'PAR': PAR,
            'time': t_points,
            'amplitude': A,
            'peak_hight': peak_hights,
            'baseline': B,
            'Tau': Tau,
            'rate': 1 / Tau,
            'flux':(1 / Tau) * A,
            'r_squared': R2,
            'traces': {"absorbance": absorbance_vs_time},
            "simulation_meta_info": meta_info_for_sim_of_this_protocol # this is the accumulated protocol
        })
    
    results = pd.DataFrame(results)

    # print some useful information
    print(f"dt = {dt}, {int((len(protocol)-1)/2)} relaxation windows fitted for {len(PAR_levels)} PAR levels.")

    if show_abs_traces:
        ax_all.legend()
        fig_all.show()

    return results


In [ ]:
def analyse_PIRK_from_simulation(c_given, protocol: Iterable, input_is_in_ms=False,
                                 fit_timeframe=np.nan, visualize_fits="",
                                 protocol_time_is_accumulated=False, use_fixed_fit_wndw_len=True, collapse_rows_to_lists = False, verbose = False):
    """
    Analyse PIRK simulation output similarly to in-vivo PIRK analysis.

    Parameters:
        c: DataFrame from simulation (e.g. from `PIRK()`), index must be time.
        protocol: list of (duration, PFD) tuples used in the simulation.
        fit_timeframe: how many milliseconds (or seconds if time_is_in_ms=False)
                       from the start of each dark window to use for fitting.

    Returns:
        DataFrame with ECS and P700 relaxation parameters for each dark window.
    """

    c = c_given.copy()

    # Fix time units
    if not input_is_in_ms: # then make it in ms.
        c.index = c.index * 1000 
        timevalues, pfds = zip(*protocol)
        protocol = list(zip(np.array(timevalues)*1000, pfds))

    # Protocol time construction
    if not protocol_time_is_accumulated:
        durations, pfds = zip(*protocol)
        onset_times = list(it.accumulate(durations))
        protocol = list(zip(onset_times, pfds))
    else:
        onset_times, pfds = zip(*protocol)

    PAR = protocol[0][1]

    # Identify "sandwiched" dark windows = true relaxation periods # remember: t indicates up to which time a certain light is applied
    relaxation_periods = []
    for i, (time, pfd) in enumerate(protocol):
        if pfd == 0:
            relaxation_periods.append((i, onset_times[i-1], time))

    if verbose: print(f"relaxation_periods:{relaxation_periods}")

    if use_fixed_fit_wndw_len:
        if fit_timeframe is np.nan:
            durations = [b-a for _,a,b in relaxation_periods]
            max_time_to_consider_for_fit = min(durations)
        else:
            max_time_to_consider_for_fit = fit_timeframe
    else:
        max_time_to_consider_for_fit = np.inf

    if verbose: print(f"Using shortest protocol dark window: {max_time_to_consider_for_fit}")

    results = []

    for idx, t_start, t_end in relaxation_periods:

        #prepare the axes for fitting
        last_time_point_fit =  min(t_start+max_time_to_consider_for_fit, t_end)
        x_raw = c.loc[t_start:last_time_point_fit,].index

        if len(x_raw) < 5:
            continue  # skip short windows

        # Interpolated fine time axis
        x_even = np.linspace(min(x_raw), max(x_raw), 1000)

        fit_done = { #these will be the rows
            "relaxation_index": idx / 2,
            "start_time_ms": t_start,
            "end_time_ms": t_end,
            "duration": t_end - t_start
        }

        for species, label, scaling, shortname in [
            ("dpsi", r'$\Delta \psi$', 1e6, "ECS"),
            ("rel_P700+", "P700$^+$", 1, "P700")
        ]:
            y_raw = scaling * c[species].loc[t_start:last_time_point_fit].to_numpy()

            interp_func = interpolate.interp1d(x_raw, y_raw, kind='linear', fill_value="extrapolate")
            y_even = interp_func(x_even)

            x_for_fit = np.array(x_even)-min(x_even) #set start to 0
            y_for_fit = y_even

            # Fit exponential decay
            A, B, Tau, R2 = fit_function(x_for_fit, y_for_fit, exponential_decay,
                                         p0=[0.002, 0, 20],
                                         bounds=([0, -np.inf, 1e-2], [np.inf, np.inf, 500]),
                                         visualize=visualize_fits,
                                         title=f'{label} relaxation {idx}')

            k = 1 / Tau
            v = A * k

            fit_done.update({
                f"{shortname}_tot": A,
                f"{shortname}_b": B,
                f"{shortname}_Tau": Tau,
                f"{shortname}_rate": k,
                f"{shortname}_flux": v,
                f"{shortname}_Rsquared": R2
            })

        results.append(fit_done)

    df = pd.DataFrame(results)

    if collapse_rows_to_lists:
        df = pd.DataFrame({col: [df[col].tolist()] for col in df.columns})
        df.index = [PAR]

    return df

In [ ]:
# series = pd.Series([1,2,3,4,5,6], index = [0.2, 2, 3, 5, 6, 7])
# series[1:5.5].index

Index([2.0, 3.0, 5.0], dtype='float64')

## compare traces visually

In [ ]:
import pandas as pd

def minmax_with_baseline(traces: list[pd.Series], independent=True, baseline_indexes_seeling_and_length = (150,15), smoothing_window_length = 30, xlim = (None, None)):
    """Min-max normalize input traces independently or jointly, preserving type and index.
    because of the noise in the measurement beam, I smooth out the curve with the savgol_filter and then obtain my values. The max I will here define as the start baseline as this way I get them all on the same level, just as in the measurement.
    the window used takes a maximum value of "time_to_dark" (see below), and then goes back a certain amount.

    IMPORTANT:
    * Baseline_index_seeling has to have the same unit (seconds or ms) as the indeces of the traces
    * Baseline_length has to be in number of pulses/measurements
    * smoothing_window_length is in number of pulses/measurements
    """
    # No longer used lines:
    # traces_as_nparrays = [np.asarray(trace, dtype="object") for trace in shorter_traces]
    # output_traces = cast_to_original_type(things_to_cast = output_nparrays, orig_types = traces)

    def max_baseline(ndarray, a, b):
        print(f"detected baseline window ranging from index {a} to {b}")
        slice_data = ndarray[a:b]
        return np.nanmean(slice_data)
    
    def min_(trace):
        return np.nanmin(trace)

    output_traces = []

    length_baseline, window_lenght = baseline_indexes_seeling_and_length

    if independent:
        for t in traces:
            # fill some NAs that can come from exlucding artifacts by mean of the three following values. NAs are otherwise not handeled well by savgol_filter and the edges are especially sensitive here
            t_filled = np.where(np.isnan(t), np.array([np.nanmean(t[i+1:i+4]) if i+1 < len(t) else np.nanmean(t[max(0, len(t)-3):]) for i in range(len(t))]), t)
            smoothed_ndarray = savgol_filter(t_filled, window_length=smoothing_window_length, polyorder=3, mode = "mirror")
            max_index = np.searchsorted(t.index, length_baseline, side='left') # get last index of values <= length_baseline
            start_index = max(0, max_index - window_lenght) #get the starting index such that the considered window is about of size window_length
            # returns ndarray and looses temporal info (index). we only use it for min & max values though, so not relevant
            t_max = max_baseline(smoothed_ndarray, start_index, max_index+1)
            t_min = min_(smoothed_ndarray)
            output_traces.append((t - t_min) / (t_max - t_min))
    else: # if dependent
        # Joint (dependent) normalization
        raise ValueError("this block of code was not fully developed and is not functional")
        all_traces = np.concatenate(traces)
        smoothed_ndarray = savgol_filter(all_traces, window_length=11, polyorder=3, mode = "mirror")
        max_index = np.searchsorted([traces[1]].index, length_baseline, side='left') # get last index of values <= length_baseline
        start_index = max(0, max_index - window_lenght) #get the starting index such that the considered window is about of size window_length
        t_global_max = max_baseline(smoothed_ndarray, start_index, max_index+1)
        t_global_min = min_(smoothed_ndarray)
        output_traces = [(t - t_global_min) / (t_global_max - t_global_min) for t in traces]

    shorter_output_traces = [trace[(xlim[0] if xlim[0] else trace.index[0]):(xlim[1] if xlim[1] else trace.index[-1])] for trace in output_traces]

    return shorter_output_traces

: 

# sensitivity analyses

In [1]:
def careful_change_parameter(s, param, value, original_parameters): # When you change a parameter in s, the parameter in m is also changed. Therefore always reset all parameters to the original ones before changing a parameter
        s.update_parameters(original_parameters)  # Reset to original parameters
        s.update_parameter(param, value)  # Change the parameter to the new value